# Controlled orthogonal-head run

Clones `Priyanshu-5257/failed_hypothesis` branch `kaggle/controlled-ortho` and runs the matched concat-mixer experiment from the repo. Training code is not in this notebook.

In [ ]:
import os, sys, platform, subprocess, json, shutil
from pathlib import Path

print("python", sys.version)
print("platform", platform.platform())
import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("capability", torch.cuda.get_device_capability(0))


In [ ]:
REPO = "https://github.com/Priyanshu-5257/failed_hypothesis.git"
BRANCH = "kaggle/controlled-ortho"
REPO_DIR = Path("/kaggle/working/repo")
WORK = Path("/kaggle/working")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.check_call(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, str(REPO_DIR)])
subprocess.check_call(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"])
subprocess.check_call(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"])


In [ ]:
%pip -q install pytest


In [ ]:
def run(cmd, cwd=None):
    print("+", *cmd, flush=True)
    p = subprocess.Popen(
        cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if rc != 0:
        raise SystemExit(rc)

run([sys.executable, "-m", "pytest", "tests", "-q"], cwd=str(REPO_DIR))


In [ ]:
variants = ("none", "act", "wv")
run_dirs = []
for ortho in variants:
    out = WORK / f"outputs_{ortho}"
    run_dirs.append(str(out))
    run(
        [
            sys.executable, "-m", "ortho_attn.train",
            "--data", str(REPO_DIR / "input.txt"),
            "--output-dir", str(out),
            "--ortho", ortho,
            "--max-iters", "5000",
            "--eval-interval", "100",
            "--eval-iters", "200",
        ],
        cwd=str(REPO_DIR),
    )


In [ ]:
report_path = WORK / "comparison.json"
run(
    [sys.executable, "-m", "ortho_attn.compare", "--runs", *run_dirs, "--out", str(report_path)],
    cwd=str(REPO_DIR),
)
report = json.loads(report_path.read_text())
print(json.dumps(report, indent=2))
assert report["ok"] and report["matched_param_count"]
(WORK / "ok.json").write_text(json.dumps({"ok": True, "branch": BRANCH, "report": report}, indent=2))
